# Pipeline test for the KuKi LLM experiments

Runs every script end-to-end on **synthetic Label Studio exports** with **mock models**
(random but schema-valid answers). No GPU needed. Everything is written to `<repo>/test_run/`;
real data in `<repo>/data/` is never touched.

Numbers produced here are meaningless; the test checks that files, joins and metrics work.
Always run in a **fresh kernel**, top to bottom. Real data and real models: `test_experiments.ipynb`.

In [ ]:
import json
import os
import shutil
import sys

# Test mode must be switched on before config is imported; config reads it only once per kernel.
if "config" in sys.modules:
    raise RuntimeError("config is already imported in this kernel: restart the kernel and run from the top")
os.environ["KUKI_TEST"] = "1"
import config

assert config.BASE_DIR == config.ROOT / "test_run"  # never delete anything outside test_run
os.chdir(config.ROOT)  # so that %run finds the scripts
shutil.rmtree(config.BASE_DIR, ignore_errors=True)

# Dummy codebook prompts; the real wrappers.json is reused.
config.PROMPT_DIR.mkdir(parents=True)
shutil.copy(config.ROOT / "prompts" / "wrappers.json", config.PROMPT_DIR)
for layer in ["L1", "L2", "L3", "L4"]:
    for lang in ["en", "ru", "tr"]:
        (config.PROMPT_DIR / f"codebook_{layer}_{lang}.md").write_text(f"Test codebook {layer} ({lang})", encoding="utf-8")

## 1. Synthetic exports
Eight Russian articles, three annotator slots (one export per slot, like the real projects).
Coverage mimics the block design: triple, double, single and not annotated. Two special cases
from the real data: A covers a task in B's project (article 6 -> annotated by A and C), and A
annotates article 0 a second time in C's project (a self-duplicate: A's own copy is kept).

In [ ]:
def make_content(i):
    return (f"Газ и безопасность {i}\n\n"
            "Москва угрожает Европе прекращением поставок газа.\n\n"
            "Жители Киева пострадали от обстрелов.\n\n"
            "Эксперты предупреждают о кризисе.")


def span(content, text, control, label):
    start = content.index(text)
    return {"id": f"{control}-{start}", "from_name": control, "to_name": "content", "type": "labels",
            "value": {"start": start, "end": start + len(text), "text": text, "labels": [label]}}


def frames(*names):
    return {"id": "frames", "from_name": "l2_frames", "to_name": "content", "type": "choices",
            "value": {"choices": list(names)}}


def coded(content, text, literal, insider, why):
    region = span(content, text, "l4_coded", "Coded Phrase")
    fields = [{"id": region["id"], "from_name": name, "to_name": "content", "type": "textarea",
               "value": {"start": region["value"]["start"], "end": region["value"]["end"], "text": [value]}}
              for name, value in [("l4_literal", literal), ("l4_insider", insider), ("l4_why", why)]]
    return [region] + fields


def annotation(annotator, content, i):
    # Even articles: all annotators agree on 'Economic' and a 'Call' span -> human alpha is defined.
    shared = [frames("Economic")] if i % 2 == 0 else []
    shared += [span(content, "о кризисе", "l3_persuasion", "Call")] if i % 2 == 0 else []
    return shared + own_annotation(annotator, content)


def own_annotation(annotator, content):
    if annotator == "A":
        return [frames("Security & defense", "Political"),
                span(content, "Москва", "l1_roles", "ANTAGONIST"),
                span(content, "Европе", "l1_roles", "INNOCENT"),
                span(content, "угрожает Европе", "l3_persuasion", "Manipulative Wording"),
                *coded(content, "прекращением поставок газа", "stopping gas", "energy as weapon", "context")]
    if annotator == "B":
        return [frames("Security & defense"),
                span(content, "Москва", "l1_roles", "ANTAGONIST"),
                span(content, "Жители Киева", "l1_roles", "INNOCENT"),
                span(content, "Эксперты предупреждают", "l3_persuasion", "Justification")]
    return [frames("Security & defense", "External regulation & reputation"),
            span(content, "Европе", "l1_roles", "INNOCENT"),
            span(content, "пострадали от обстрелов", "l3_persuasion", "Manipulative Wording")]


COVERAGE = ["ABC", "ABC", "AB", "BC", "A", "B", "C", ""]  # slot projects that annotate each article
EXTRA = {"B": {6: "A"}, "C": {0: "A"}}                     # slot -> {article: other annotator working there}
USER_IDS = {letter: user for user, letter in config.ANNOTATOR_IDS["ru"].items()}
SOURCES = ["ria_novosti", "theinsider"]

raw_dir = config.RAW_DIR
raw_dir.mkdir(parents=True)
for slot in "ABC":
    tasks = []
    for i, covered_by in enumerate(COVERAGE):
        content = make_content(i)
        annotators = ([slot] if slot in covered_by else []) + ([EXTRA[slot][i]] if i in EXTRA.get(slot, {}) else [])
        annotations = [{"id": 10 * i + n, "completed_by": USER_IDS[a], "result": annotation(a, content, i),
                        "was_cancelled": False} for n, a in enumerate(annotators)]
        tasks.append({"id": 100 + i, "annotations": annotations,
                      "data": {"content": content, "source": SOURCES[i % 2], "author": f"Автор {i % 3}"}})
    (raw_dir / f"ru_{slot}.json").write_text(json.dumps(tasks, ensure_ascii=False), encoding="utf-8")
sorted(p.name for p in raw_dir.iterdir())

## 2. Structural check of an export
Run this on a **real** export first (e.g. `Path("data/raw/ru_A.json")`) to confirm field
names before trusting `prep_00`: task data keys, control names, annotation counts.

In [ ]:
from collections import Counter


def inspect_export(path):
    tasks = json.loads(path.read_text(encoding="utf-8"))
    print(f"{path.name}: {len(tasks)} tasks")
    print("task data keys:", sorted(tasks[0]["data"].keys()))
    print("annotations per task:", Counter(len(t.get("annotations", [])) for t in tasks))
    print("Label Studio users (completed_by):", Counter(str(a["completed_by"]) for t in tasks for a in t.get("annotations", [])))
    controls = Counter(r["from_name"] for t in tasks for a in t.get("annotations", []) for r in a["result"])
    print("controls:", dict(controls))


inspect_export(raw_dir / "ru_A.json")

## 3. Data preparation and splits

In [ ]:
%run prep_00_parse_exports.py --langs ru

In [ ]:
from llm_io import read_jsonl

# Expected: article 0 (self-duplicate) annotated by A, B, C once each; article 6 by ru_A and ru_C.
for a in read_jsonl(config.PREP_DIR / "articles.jsonl"):
    print(a["content"].splitlines()[0], a["annotators"])

In [ ]:
import pandas as pd
import config
from llm_io import read_jsonl

articles = read_jsonl(config.PREP_DIR / "articles.jsonl")
print("entities of article 0:", articles[0]["entities"])
labels = pd.read_csv(config.PREP_DIR / "human_labels.csv")
# Expected: Москва/Европе/Жители Киева as entities; frames and paragraph labels per annotator.
labels[labels["value"] == 1].head(12)

In [ ]:
%run prep_01_make_splits.py --n-dev 1 --n-l4 2

## 4. Prompt check
What the model actually sees: one paragraph-with-context call, native prompt language.

In [ ]:
import llm_io

article = llm_io.load_split("s1_main_grid")[0]
for message in llm_io.build_messages(article, "L3", "para_ctx", "native", target=2):
    print(f"--- {message['role']} ---\n{message['content']}\n")
print(json.dumps(llm_io.output_schema("L1", llm_io.entity_names(article)), ensure_ascii=False)[:300])

## 5. Stage 1: main grid (mock models)
Two mock models give two sizes and two profiles, so the decomposition has something to split.

In [ ]:
%run s1_main_grid.py --model mock-M --dev
%run s1_main_grid.py --model mock-M
%run s1_main_grid.py --model mock-S

In [ ]:
# Re-running must resume: expect '0 to run'.
%run s1_main_grid.py --model mock-S

In [ ]:
# One prediction file per experiment and model.
print(sorted(p.name for p in config.PRED_DIR.iterdir()))
predictions = pd.DataFrame(read_jsonl(config.PRED_DIR / "s1_main_grid__mock-M.jsonl") + read_jsonl(config.PRED_DIR / "s1_main_grid__mock-S.jsonl"))
print(predictions.groupby(["model", "layer", "granularity", "prompt_lang"]).size().unstack())
predictions[["key", "raw", "parse_ok"]].head(3)

In [ ]:
%run evaluate.py --experiment s1_main_grid

Expected here: a `SingularMatrixWarning`, because in the mock grid size and profile are confounded (mock-S = S + english, mock-M = M + multilingual). The real grid crosses them.

In [ ]:
%run analyze_s1_decomposition.py

## 6. Stages 2-5 (mock models)

In [ ]:
%run s2_stability.py --model mock-M --granularity para_ctx --prompt-lang en --n-samples 3
%run evaluate.py --experiment s2_stability

In [ ]:
%run s3_metadata_probe.py --model mock-M
%run evaluate.py --experiment s3_metadata_probe
pd.read_csv(config.RESULT_DIR / "s3_metadata_probe_prevalence_by_meta.csv").head()

In [ ]:
%run s4_generalisation.py --model mock-S --granularity doc --prompt-lang en
%run evaluate.py --experiment s4_generalisation

In [ ]:
%run s5_l4_pilot.py --model mock-M
%run evaluate.py --experiment s5_l4_pilot